# Intermediate Python Exercises: Part 2 (Exercises 41-65 of 65)

Continues directly from Part 1 — multiple inheritance, composition, the Singleton pattern, dataclasses, CSV/JSON/regex processing, robust error handling, threading, and several mini-projects (password checker, guessing game, to-do list, Caesar cipher, pattern printing).

*Adapted for practice from the exercise list at [PYnative](https://pynative.com/intermediate-python-exercises/).*

---

**Note:** Exercise 51 (API requests with the `requests` library) is a network-dependent exercise and is described but not executed automatically in this notebook — see the note in that section.


## Exercise 41. Multiple Inheritance and MRO

**Concept:** inheriting from two parents, Method Resolution Order

**Problem:** Create Flyer and Swimmer classes, then a Duck that inherits from both, and inspect the search order Python uses.

**Given:**
```
class Duck(Flyer, Swimmer)
```

**Expected Output:**
```
Flying high!
Swimming fast!
MRO: (Duck, Flyer, Swimmer, object)
```

**Hint:** List both parent classes in the class definition; check Duck.__mro__ to see the lookup order.

In [ ]:
class Flyer:
    def fly(self):
        print("Flying high!")

class Swimmer:
    def swim(self):
        print("Swimming fast!")

class Duck(Flyer, Swimmer):
    pass

d = Duck()
d.fly()
d.swim()

print(f"MRO: {Duck.__mro__}")

**Explanation:** Duck inherits every method from both Flyer and Swimmer. __mro__ reveals the exact order Python searches through parent classes when looking up a method — every class ultimately ends at the built-in object class.

## Exercise 42. Composition Over Inheritance

**Concept:** 'has-a' relationships instead of 'is-a'

**Problem:** Build a Computer class that's composed of separate CPU and RAM objects rather than inheriting from them.

**Given:**
```
Computer("Intel i7", "16GB")
```

**Expected Output:**
```
Computer with Intel i7 CPU and 16GB RAM.
```

**Hint:** Instantiate CPU and RAM objects inside Computer's __init__, and store them as attributes.

In [ ]:
class CPU:
    def __init__(self, model):
        self.model = model

class RAM:
    def __init__(self, size):
        self.size = size

class Computer:
    def __init__(self, cpu_model, ram_size):
        self.cpu = CPU(cpu_model)
        self.ram = RAM(ram_size)

    def __str__(self):
        return f"Computer with {self.cpu.model} CPU and {self.ram.size} RAM."

my_pc = Computer("Intel i7", "16GB")
print(my_pc)

**Explanation:** Computer doesn't inherit from CPU or RAM — it simply holds references to instances of them, a 'has-a' relationship. This keeps the pieces independent: swapping in a different CPU later only touches the internal object, not a class hierarchy.

## Exercise 43. The Singleton Pattern

**Concept:** __new__, enforcing a single instance

**Problem:** Ensure a Database class can only ever have one instance — a second 'creation' should return the original.

**Given:**
```
db1 = Database(); db2 = Database()
```

**Expected Output:**
```
Are they the same instance? True
```

**Hint:** Store the one instance in a class-level variable and check it inside __new__ before creating a new object.

In [ ]:
class Database:
    _instance = None

    def __new__(cls):
        if cls._instance is None:
            print("Loading Database (First time only)...")
            cls._instance = super(Database, cls).__new__(cls)
        return cls._instance

db1 = Database()
db2 = Database()

print(f"Are they the same instance? {db1 is db2}")

**Explanation:** __new__ is the method actually responsible for creating an object, running before __init__. By checking cls._instance first, the real object-creation step only ever happens once; every later call just returns that same stored instance, which is why `is` (identity) reports True.

## Exercise 44. Data Classes for Boilerplate-Free Objects

**Concept:** @dataclass, auto-generated __init__/__repr__/__eq__

**Problem:** Turn a plain Book class into a @dataclass and see the automatic string representation and equality.

**Given:**
```
Book("1984", "George Orwell", 328)
```

**Expected Output:**
```
Book(title='1984', author='George Orwell', pages=328)
```

**Hint:** Import dataclass, decorate the class, and list attributes with type hints instead of writing __init__ by hand.

In [ ]:
from dataclasses import dataclass

@dataclass
class Book:
    title: str
    author: str
    pages: int

b1 = Book("1984", "George Orwell", 328)
b2 = Book("1984", "George Orwell", 328)

print(b1)
print(f"Is b1 equal to b2? {b1 == b2}")

**Explanation:** @dataclass generates __init__, __repr__, and __eq__ automatically from the type-hinted attributes, so print(b1) shows readable data instead of a memory address, and == compares field values rather than object identity.

## Exercise 45. Process a CSV of Salaries

**Concept:** csv.DictReader / DictWriter

**Problem:** Read a CSV of employee salaries, compute the average, and write the data plus that average to a new CSV.

**Given:**
```
data.csv with Name,Salary columns
```

**Expected Output:**
```
output.csv contains the original rows plus an Average row
```

**Hint:** DictReader lets you access columns by name (row['Salary']) instead of a numeric index.

In [ ]:
import csv
import io

# Using an in-memory CSV for demonstration instead of a file on disk
sample_csv = "Name,Salary\nAlice,50000\nBob,60000\nCharlie,70000\n"

def process_salaries(csv_text):
    reader = csv.DictReader(io.StringIO(csv_text))
    salaries = []
    rows = []
    for row in reader:
        salaries.append(int(row['Salary']))
        rows.append(row)

    avg_salary = sum(salaries) / len(salaries)

    output = io.StringIO()
    writer = csv.DictWriter(output, fieldnames=['Name', 'Salary'])
    writer.writeheader()
    writer.writerows(rows)
    writer.writerow({'Name': 'Average', 'Salary': avg_salary})
    return output.getvalue()

print(process_salaries(sample_csv))

**Explanation:** DictReader treats each row as a dictionary keyed by the header names, which is more robust than numeric indexing if columns get reordered. This version reads/writes an in-memory string (io.StringIO) instead of a real file, but with open(path) as f: works the same way for actual CSV files.

## Exercise 46. Parse and Update Nested JSON

**Concept:** json.loads / json.dumps, nested key access

**Problem:** Parse a JSON string, update a deeply nested value, and convert it back to a formatted JSON string.

**Given:**
```
'{"id": 1, "profile": {"name": "Alice", "settings": {"theme": "light"}}}'
```

**Expected Output:**
```
theme changed from "light" to "dark"
```

**Hint:** json.loads() turns the string into nested dictionaries you can index into and modify directly.

In [ ]:
import json

user_json = '{"id": 1, "profile": {"name": "Alice", "settings": {"theme": "light"}}}'

data = json.loads(user_json)

data['profile']['settings']['theme'] = 'dark'

updated_json = json.dumps(data, indent=4)

print("Updated Profile Settings:")
print(updated_json)

**Explanation:** json.loads (string input) parses JSON into ordinary nested Python dictionaries, so deep values are reached by chaining keys: data['profile']['settings']['theme']. json.dumps(data, indent=4) converts it back to a nicely formatted JSON string.

## Exercise 47. Extract Emails with Regex

**Concept:** re.findall, raw strings

**Problem:** Pull every valid-looking email address out of a block of text.

**Given:**
```
"Contact us at support@company.com or sales.dept@office.org for help."
```

**Expected Output:**
```
['support@company.com', 'sales.dept@office.org']
```

**Hint:** A raw string pattern like [a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,} matches typical emails.

In [ ]:
import re

text = "Contact us at support@company.com or sales.dept@office.org for help."

email_pattern = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'

emails = re.findall(email_pattern, text)

print(f"Emails found: {emails}")

**Explanation:** The r'...' raw-string prefix stops Python from interpreting backslashes specially, which regex patterns rely on. re.findall() scans the whole string and returns every non-overlapping match as a list.

## Exercise 48. Handle Multiple Exception Types

**Concept:** try/except/finally, catching specific errors

**Problem:** Write a divide function that gracefully handles division by zero and wrong argument types, always printing a completion message.

**Given:**
```
divide_numbers(10, 0); divide_numbers(10, 'apple')
```

**Expected Output:**
```
Error: Cannot divide by zero!
Operation complete.
...
Error: Please provide numbers...
Operation complete.
```

**Hint:** Stack multiple except blocks for specific error types, and use finally for code that must always run.

In [ ]:
def divide_numbers(a, b):
    try:
        result = a / b
        print(f"Result: {result}")
    except ZeroDivisionError:
        print("Error: Cannot divide by zero!")
    except TypeError:
        print("Error: Please provide numbers (integers or floats).")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
    finally:
        print("Operation complete.")

divide_numbers(10, 0)
print("-" * 20)
divide_numbers(10, "apple")

**Explanation:** Catching specific exceptions (ZeroDivisionError, TypeError) before a general Exception makes failures easier to diagnose. finally always executes, whether the try block succeeded or an exception was caught — ideal for cleanup steps.

## Exercise 49. Walk a Directory Tree for .txt Files

**Concept:** os.walk, os.path.join

**Problem:** Recursively search a directory and its subfolders for every .txt file.

**Given:**
```
a starting directory path
```

**Expected Output:**
```
prints the full path of every .txt file found
```

**Hint:** os.walk(path) yields (root, dirs, files) for every folder it visits, recursively.

In [ ]:
import os

def find_txt_files(start_dir):
    for root, dirs, files in os.walk(start_dir):
        for file in files:
            if file.endswith(".txt"):
                full_path = os.path.join(root, file)
                print(f"Found: {full_path}")

find_txt_files('.')

**Explanation:** os.walk() handles the recursion into subfolders automatically, visiting every directory beneath the starting point. os.path.join() builds paths with the correct separator for the current operating system, and .endswith('.txt') filters to just the files of interest.

## Exercise 50. Run Two Tasks Concurrently With Threading

**Concept:** threading.Thread, start()/join()

**Problem:** Simulate two 2-second downloads running at the same time instead of one after another.

**Given:**
```
two tasks that each sleep 2 seconds
```

**Expected Output:**
```
Total time taken: ~2.00 seconds (not 4)
```

**Hint:** Create Thread objects with target/args, call start() on both, then join() to wait for them to finish.

In [ ]:
import threading
import time

def download_file(name, duration):
    print(f"Starting download: {name}")
    time.sleep(duration)
    print(f"Finished download: {name}")

start = time.perf_counter()

t1 = threading.Thread(target=download_file, args=("File_A", 2))
t2 = threading.Thread(target=download_file, args=("File_B", 2))

t1.start()
t2.start()

t1.join()
t2.join()

end = time.perf_counter()
print(f"Total time taken: {end - start:.2f} seconds")

**Explanation:** start() hands each function off to run on its own thread, so both 2-second sleeps happen at the same time rather than back to back. join() blocks the main program until a thread finishes, which is why the total time is close to 2 seconds instead of 4.

## Exercise 51: API Requests with Error Handling

**Concept:** the `requests` library, `raise_for_status()`, network error handling

**Problem:** Fetch data from a public API (e.g. JSONPlaceholder) and handle connection errors or bad status codes gracefully.

**Hint:** Always pass a `timeout=` to `requests.get()`, and call `response.raise_for_status()` to turn a bad HTTP status into a catchable exception.

```python
import requests

def get_todo_data(todo_id):
    url = f"https://jsonplaceholder.typicode.com/todos/{todo_id}"
    try:
        response = requests.get(url, timeout=5)
        response.raise_for_status()
        data = response.json()
        print(f"Data: {data}")
    except requests.exceptions.HTTPError as err:
        print(f"HTTP Error: {err}")
    except requests.exceptions.ConnectionError:
        print("Error: Could not connect to the server.")
    except Exception as e:
        print(f"An error occurred: {e}")

get_todo_data(1)
```

**Explanation:** `timeout=5` stops the program from hanging forever if a server never responds. `response.json()` parses the JSON body straight into a Python dictionary. `raise_for_status()` turns HTTP error codes (4xx/5xx) into a Python exception you can catch, instead of having to check `response.status_code` manually.

*(This cell is illustrative only — it isn't auto-executed in this notebook since it requires outbound internet access. Copy it into a code cell to run it yourself.)*

## Exercise 52. Word Frequency Counter (Regex-Cleaned)

**Concept:** re.sub for cleanup + Counter

**Problem:** Count word occurrences in a paragraph, case-insensitively, with punctuation stripped out first.

**Given:**
```
"Python is great. Python is fast, and learning Python is fun!"
```

**Expected Output:**
```
{'python': 3, 'is': 3, 'great': 1, 'fast': 1, 'and': 1, 'learning': 1, 'fun': 1}
```

**Hint:** re.sub(r'[^\w\s]', '', text) strips punctuation; then .lower().split() and Counter do the rest.

In [ ]:
import re
from collections import Counter

def count_words(text):
    clean_text = re.sub(r'[^\w\s]', '', text).lower()
    words = clean_text.split()
    return dict(Counter(words))

text = "Python is great. Python is fast, and learning Python is fun!"
print(f"Word Frequency: {count_words(text)}")

**Explanation:** The regex removes any character that isn't a word character or whitespace, so 'fast,' becomes 'fast' rather than keeping the comma attached. Counter then tallies the cleaned, lowercased words in one step.

## Exercise 53. Nested-Dictionary Student Records

**Concept:** dict of dicts as an in-memory database

**Problem:** Manage student records with add/update/display functions, using student ID as the lookup key.

**Given:**
```
add ID 101 Alice/A; update ID 101 to A+
```

**Expected Output:**
```
ID: 101 | Name: Alice | Grade: A+
```

**Hint:** Use the student ID as the outer dict's key, and a small dict of name/grade as the value.

In [ ]:
students = {}

def add_student(s_id, name, grade):
    students[s_id] = {"name": name, "grade": grade}

def update_grade(s_id, new_grade):
    if s_id in students:
        students[s_id]["grade"] = new_grade
    else:
        print("Student not found.")

def display_students():
    for s_id, info in students.items():
        print(f"ID: {s_id} | Name: {info['name']} | Grade: {info['grade']}")

add_student(101, "Alice", "A")
update_grade(101, "A+")
display_students()

**Explanation:** Using the ID as the dictionary key gives near-instant lookups compared to scanning a list. students[s_id]["grade"] drills into the inner dictionary to update just that field, and the `in` check prevents crashing on an unknown ID.

## Exercise 54. Password Strength Scorer

**Concept:** any() with a generator, boolean summing

**Problem:** Score a password 0-5 based on length, uppercase, lowercase, digit, and special-character criteria.

**Given:**
```
"P@ssword123"
```

**Expected Output:**
```
Strength: 5/5 (Very Strong)
```

**Hint:** Build a list of True/False checks and sum() it — True counts as 1, False as 0.

In [ ]:
def check_password(password):
    checks = [
        len(password) >= 8,
        any(char.isupper() for char in password),
        any(char.islower() for char in password),
        any(char.isdigit() for char in password),
        any(not char.isalnum() for char in password)
    ]
    score = sum(checks)
    levels = {5: "Very Strong", 4: "Strong", 3: "Moderate", 2: "Weak", 1: "Very Weak", 0: "Invalid"}
    return f"Strength: {score}/5 ({levels.get(score)})"

print(check_password("P@ssword123"))

**Explanation:** any(condition for char in password) checks whether at least one character satisfies a rule, without writing a manual loop. not char.isalnum() flags anything that's neither a letter nor a digit as a special character. Summing the five booleans gives the final score directly.

## Exercise 55. Number Guessing Game

**Concept:** random.randint, while loop, input validation

**Problem:** The program picks a random number 1-100; the user gets 10 guesses with higher/lower feedback.

**Given:**
```
target picked randomly, up to 10 guesses
```

**Expected Output:**
```
feedback of Higher!/Lower! until guessed or out of attempts
```

**Hint:** Wrap int(input()) in try/except so non-numeric input doesn't crash the game.

In [ ]:
import random

def play_game():
    target = random.randint(1, 100)
    attempts = 10

    print("I'm thinking of a number between 1 and 100.")

    while attempts > 0:
        try:
            guess = int(input(f"({attempts} left) Enter guess: "))
        except ValueError:
            print("Invalid input. Enter a number!")
            continue

        if guess == target:
            print(f"Correct! The number was {target}.")
            return
        elif guess < target:
            print("Higher!")
        else:
            print("Lower!")

        attempts -= 1

    print(f"Game Over. The number was {target}.")

# play_game() left uncommented would wait for interactive input
print("play_game() defined — call it interactively to play.")

**Explanation:** random.randint(1, 100) includes both endpoints. Wrapping int(input()) in try/except stops a non-numeric guess from crashing the program — continue simply re-prompts. return exits the function immediately the moment the correct number is guessed.

## Exercise 56. File Statistics (Lines, Words, Characters)

**Concept:** line-by-line file iteration

**Problem:** Count the lines, words, and characters in a text file.

**Given:**
```
a file containing "Hello World\nPython is fun."
```

**Expected Output:**
```
Lines: 2 | Words: 5 | Characters: 27
```

**Hint:** Iterate the file object directly to get one line at a time; use .split() for words and len() for characters.

In [ ]:
def file_stats_from_text(text):
    lines, words, chars = 0, 0, 0
    for line in text.splitlines(keepends=True):
        lines += 1
        words += len(line.split())
        chars += len(line)
    print(f"Lines: {lines} | Words: {words} | Characters: {chars}")

# Demonstrated on an in-memory string; with a real file use:
# with open(filename, 'r') as f:
#     for line in f: ...
sample_text = "Hello World\nPython is fun.\n"
file_stats_from_text(sample_text)

**Explanation:** Reading a file line by line (for line in f) avoids loading the whole file into memory at once. len(line) includes the trailing newline character, matching how most text editors report character counts.

## Exercise 57. Prime Number Generator (Optimized)

**Concept:** square-root divisor bound

**Problem:** List all primes within a range, checking divisors only up to each number's square root.

**Given:**
```
range 10 to 50
```

**Expected Output:**
```
[11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47]
```

**Hint:** If n has a factor, at least one of its factors is <= sqrt(n), so you never need to check beyond that.

In [ ]:
def is_prime(n):
    if n < 2:
        return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False
    return True

def generate_primes(start, end):
    return [num for num in range(start, end + 1) if is_prime(num)]

primes = generate_primes(10, 50)
print(f"Primes in range: {primes}")

**Explanation:** Checking divisors only up to int(n**0.5) + 1 is a well-known optimization: any factor larger than the square root would have to pair with one smaller than it, which would already have been caught. The list comprehension then filters the whole range through this helper.

## Exercise 58. Validate an Email With Regex

**Concept:** anchored regex patterns (^ and $)

**Problem:** Check whether an email address matches a valid username@domain.extension structure.

**Given:**
```
"python_pro@gmail.com" vs "bad-email@com"
```

**Expected Output:**
```
'python_pro@gmail.com' is a Valid Email
'bad-email@com' is an Invalid Email
```

**Hint:** ^ and $ anchor the match to the whole string; \w{2,3} requires a plausible extension length.

In [ ]:
import re

def validate_email(email):
    regex = r'^[a-z0-9]+[\._]?[a-z0-9]+[@]\w+[.]\w{2,3}$'
    if re.search(regex, email):
        print(f"'{email}' is a Valid Email")
    else:
        print(f"'{email}' is an Invalid Email")

validate_email("python_pro@gmail.com")
validate_email("bad-email@com")

**Explanation:** ^ and $ ensure the entire string must fit the pattern, not just some substring of it. Requiring \w{2,3} after a literal dot demands something that looks like a real top-level domain, which is why 'bad-email@com' — missing that dot-extension — fails the check.

## Exercise 59. File-Based To-Do List

**Concept:** append vs write file modes, persistence

**Problem:** Build add/view/clear functions for a to-do list that's saved to a text file so it survives between runs.

**Given:**
```
add_task("Buy Milk"); add_task("Finish Python Project")
```

**Expected Output:**
```
--- Current Tasks ---
1. Buy Milk
2. Finish Python Project
```

**Hint:** Open in 'a' (append) mode to add without erasing; open in 'w' mode and close immediately to clear the file.

In [ ]:
FILE_NAME = "todo.txt"

def add_task(task):
    with open(FILE_NAME, "a") as f:
        f.write(task + "\n")

def view_tasks():
    try:
        with open(FILE_NAME, "r") as f:
            print("\n--- Current Tasks ---")
            for i, line in enumerate(f, 1):
                print(f"{i}. {line.strip()}")
    except FileNotFoundError:
        print("No tasks found.")

def clear_tasks():
    open(FILE_NAME, "w").close()
    print("List cleared.")

clear_tasks()
add_task("Code in Python")
add_task("Walk the dog")
view_tasks()

**Explanation:** "a" mode appends new text after whatever's already in the file, unlike "w" mode which wipes it first. .strip() removes the trailing newline each line carries when read back, so the printed list looks clean.

## Exercise 60. Cryptographically Secure Password Generator

**Concept:** secrets module vs random

**Problem:** Generate a random password of a given length using a mix of letters, digits, and punctuation.

**Given:**
```
length = 16
```

**Expected Output:**
```
a 16-character password mixing letters, digits, and symbols
```

**Hint:** Combine string.ascii_letters, string.digits, and string.punctuation into one character pool.

In [ ]:
import secrets
import string

def generate_password(length):
    if length < 4:
        return "Length too short!"
    pool = string.ascii_letters + string.digits + string.punctuation
    password = "".join(secrets.choice(pool) for _ in range(length))
    return password

print(f"Secure Password: {generate_password(16)}")

**Explanation:** secrets.choice() is designed for security-sensitive randomness, unlike random.choice(), which is predictable enough to be unsuitable for passwords. "".join(...) glues the individually chosen characters into a single password string.

## Exercise 61. Generate Pascal's Triangle

**Concept:** building each row from the previous one

**Problem:** Generate the first N rows of Pascal's Triangle, where each interior number is the sum of the two above it.

**Given:**
```
5 rows
```

**Expected Output:**
```
[1]
[1, 1]
[1, 2, 1]
[1, 3, 3, 1]
[1, 4, 6, 4, 1]
```

**Hint:** Every row starts and ends with 1; the middle values are prev_row[j-1] + prev_row[j].

In [ ]:
def generate_pascal(n):
    triangle = [[1]]
    for i in range(1, n):
        prev_row = triangle[-1]
        new_row = [1]
        for j in range(1, i):
            new_row.append(prev_row[j-1] + prev_row[j])
        new_row.append(1)
        triangle.append(new_row)
    return triangle

for row in generate_pascal(5):
    print(row)

**Explanation:** triangle[-1] grabs the most recently generated row to build from. Each interior value is the sum of the two values above it in the previous row; the 1s at each end are added directly rather than computed.

## Exercise 62. Auto-Recalculating Property

**Concept:** @property that reflects current state

**Problem:** Give a Circle an area property that automatically updates whenever radius changes, with no manual recalculation step.

**Given:**
```
c = Circle(5); then c.radius = 10
```

**Expected Output:**
```
Area with radius 5: 78.54
Area with radius 10: 314.16
```

**Hint:** Because area is computed inside a @property method, it always uses whatever self.radius currently is.

In [ ]:
import math

class Circle:
    def __init__(self, radius):
        self.radius = radius

    @property
    def area(self):
        return math.pi * (self.radius ** 2)

c = Circle(5)
print(f"Area with radius 5: {c.area:.2f}")

c.radius = 10
print(f"Area with radius 10: {c.area:.2f}")

**Explanation:** area isn't stored — it's recalculated fresh every time c.area is accessed, based on whatever self.radius happens to be at that moment. Since no setter was defined for area, users can read it but can't directly overwrite it with a bogus value.

## Exercise 63. Caesar Cipher

**Concept:** ord()/chr() with modulo wraparound

**Problem:** Encrypt a message by shifting each letter a fixed number of positions in the alphabet, wrapping Z back to A.

**Given:**
```
"Hello World!", shift 3
```

**Expected Output:**
```
"Khoor Zruog!"
```

**Hint:** For each letter, the new position is (original_position + shift) % 26.

In [ ]:
def caesar_cipher(text, shift):
    result = ""
    for char in text:
        if char.isalpha():
            start = ord('A') if char.isupper() else ord('a')
            new_pos = (ord(char) - start + shift) % 26
            result += chr(start + new_pos)
        else:
            result += char
    return result

encrypted = caesar_cipher("Hello World!", 3)
print(f"Encrypted: {encrypted}")
print(f"Decrypted: {caesar_cipher(encrypted, -3)}")

**Explanation:** ord(char) - start converts a letter to a 0-25 index within its case's alphabet. % 26 wraps the shifted index back around from Z to A instead of running off the end. Passing a negative shift reverses the process, decrypting the message.

## Exercise 64. Hollow Triangle Pattern

**Concept:** boundary-only printing in nested loops

**Problem:** Print a triangle outline of height N made only of its edges — the left slope, the right slope, and the base — leaving the interior empty.

**Given:**
```
height = 5
```

**Expected Output:**
```
    *
   * *
  *   *
 *     *
*********
```

**Hint:** On each row, print a star only at the first column, the last column of that row, or on the final (base) row.

In [ ]:
def hollow_triangle(n):
    for i in range(n):
        for j in range(n - i - 1):
            print(" ", end="")
        for k in range(2 * i + 1):
            if i == n - 1 or k == 0 or k == 2 * i:
                print("*", end="")
            else:
                print(" ", end="")
        print()

hollow_triangle(5)

**Explanation:** The leading spaces (n - i - 1 of them) push each row's triangle portion into alignment. Within the triangle portion, a star only appears at the row's first position (k == 0), its last position (k == 2*i), or anywhere on the bottom row (i == n - 1); everywhere else prints a blank space, leaving the interior hollow.

## Exercise 65. Solid Diamond Pattern

**Concept:** two mirrored triangles (growing then shrinking)

**Problem:** Print a solid diamond of stars: a triangle that grows to its widest point, then shrinks back down symmetrically.

**Given:**
```
size = 5
```

**Expected Output:**
```
    *
   ***
  *****
 ******* 
*********
 ******* 
  *****
   ***
    *
```

**Hint:** Print a growing half (1 up to n stars, odd counts) then a shrinking mirror image of it.

In [ ]:
def solid_diamond(n):
    # Top half (including the middle row)
    for i in range(n):
        spaces = " " * (n - i - 1)
        stars = "*" * (2 * i + 1)
        print(spaces + stars)
    # Bottom half (mirrors the top, excluding the middle row)
    for i in range(n - 2, -1, -1):
        spaces = " " * (n - i - 1)
        stars = "*" * (2 * i + 1)
        print(spaces + stars)

solid_diamond(5)

**Explanation:** Each row prints 2*i + 1 stars (an odd count), which is what gives a diamond its pointed top and bottom rather than flat edges. The bottom loop counts back down from n - 2, mirroring every row of the top half except the widest one, which appears only once.